# Deliverable 1: Data Analysis for Gait-Based Parkinson's Disease Detection

**Scope.** This notebook implements only the analysis requested in `Deliverable_1_List_of_Task.pdf`: dataset description, patient/control signal examples, TSFresh feature extraction, per-subject feature statistics, and PCA/t-SNE plots for right foot, left foot, and concatenated feet.

**Excluded.** No classifier, baseline-model, validation-performance, or test-performance results are included here.

## Analysis Decisions

- The full demographics spreadsheet is shown first, before restricting to the subjects present in the provided folds.
- The folds contain the same 100 unique walk-01 subjects under different train/validation/test assignments. Dataset-level signal and feature analysis therefore uses each subject once to avoid duplicate counting.
- The required demographic variables (`Group`, `Age`, and `Gender`) are audited for missing values before reporting. Optional fields with missing values are documented but are not imputed because they are not required for this deliverable and are not used in the projections.
- TSFresh features are calculated with the required suggested configuration: 5-second windows and a 2.5-second step.
- Two preprocessing variants are compared: no endpoint removal and removal of 5 seconds at both recording endpoints. The trimming is **not required by the deliverable PDF** and was **not taken as a rule from a reviewed paper**. It is tested here as a sensitivity analysis because the initial and final seconds of a walking recording may include acceleration, turning, slowing, or stopping rather than stable repeated gait.
- Zero-variance feature columns are removed, then z-score normalization is applied before every PCA and t-SNE calculation.

In [ ]:
from pathlib import Path
import os
import warnings

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tsfresh import extract_features
from tsfresh.feature_extraction import MinimalFCParameters
from tsfresh.utilities.dataframe_functions import impute

warnings.filterwarnings('ignore', category=RuntimeWarning)
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path.cwd()
SPLITS_DIR = ROOT / 'splits'
DEMOGRAPHICS_PATH = ROOT / 'demographics.xls'
OUTPUT_DIR = ROOT / 'deliverable1_notebook_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

assert SPLITS_DIR.exists(), 'Run this notebook from the Data project directory.'
assert DEMOGRAPHICS_PATH.exists(), 'demographics.xls was not found in the project directory.'

SAMPLE_RATE = 100
WINDOW_SECONDS = 5.0
STEP_SECONDS = 2.5
WINDOW_SAMPLES = int(WINDOW_SECONDS * SAMPLE_RATE)
STEP_SAMPLES = int(STEP_SECONDS * SAMPLE_RATE)
TRIM_SECONDS = 5.0
RECOMPUTE_FEATURES = False

SIGNAL_COLUMNS = ['Time'] + [f'L{i}' for i in range(1, 9)] + [f'R{i}' for i in range(1, 9)] + ['TotalLeft', 'TotalRight']
LEFT_KINDS = [f'L{i}' for i in range(1, 9)] + ['TotalLeft']
RIGHT_KINDS = [f'R{i}' for i in range(1, 9)] + ['TotalRight']
CHANNEL_INDEX = {name: idx for idx, name in enumerate(SIGNAL_COLUMNS)}
PALETTE = {'Control': '#2F6F9F', 'Patient': '#B24545'}

print(f'Window length: {WINDOW_SECONDS:.1f} s ({WINDOW_SAMPLES} samples)')
print(f'Step size: {STEP_SECONDS:.1f} s ({STEP_SAMPLES} samples)')

## 1. Full Demographics Spreadsheet Distribution

The spreadsheet describes the full PhysioNet collection available with the project files. Null values are audited before any subset is created.

In [ ]:
demographics_raw = pd.read_excel(DEMOGRAPHICS_PATH, engine='xlrd')
demographics = demographics_raw.copy()
demographics['Group'] = demographics['Group'].replace({'CO': 'Control', 'PD': 'Patient'})
demographics['Gender'] = demographics['Gender'].astype(str).str.strip().str.lower().replace({'male': 'Male', 'female': 'Female'})

required_demographic_fields = ['ID', 'Group', 'Age', 'Gender']
null_audit = demographics.isna().sum().rename('missing_values').to_frame()
null_audit['percent_missing'] = (100 * null_audit['missing_values'] / len(demographics)).round(1)
null_audit['used_for_required_summary'] = null_audit.index.isin(required_demographic_fields)
null_audit['handling'] = np.where(
    null_audit['used_for_required_summary'],
    'Complete required field; use as reported',
    'Not required for this deliverable; retain NA and exclude from analysis'
)

print(f'Full spreadsheet: {len(demographics)} subjects')
print('Missing-value audit:')
display(null_audit[null_audit['missing_values'] > 0])
null_audit.to_csv(OUTPUT_DIR / 'demographics_null_audit.csv')
assert demographics[required_demographic_fields].isna().sum().sum() == 0
print('The required demographic fields contain no null values; no imputation is needed for the required summary.')

In [ ]:
def demographic_summary(frame):
    return (frame.groupby('Group', as_index=False)
            .agg(subjects=('ID', 'count'),
                 age_mean=('Age', 'mean'),
                 age_std=('Age', 'std'),
                 male=('Gender', lambda s: int((s == 'Male').sum())),
                 female=('Gender', lambda s: int((s == 'Female').sum())))
            .round({'age_mean': 1, 'age_std': 1}))

full_demographic_summary = demographic_summary(demographics)
full_demographic_summary.to_csv(OUTPUT_DIR / 'full_demographic_summary.csv', index=False)
print('Required demographic summary for the complete spreadsheet:')
display(full_demographic_summary)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
sns.countplot(data=demographics, x='Group', hue='Group', palette=PALETTE, legend=False, ax=axes[0])
axes[0].set_title('Subjects by diagnosis')
axes[0].set_ylabel('Number of subjects')
sns.countplot(data=demographics, x='Group', hue='Gender', palette=['#204251', '#8C9FB1'], ax=axes[1])
axes[1].set_title('Sex distribution by diagnosis')
axes[1].set_ylabel('Number of subjects')
sns.boxplot(data=demographics, x='Group', y='Age', hue='Group', palette=PALETTE, legend=False, ax=axes[2])
sns.stripplot(data=demographics, x='Group', y='Age', color='#204251', alpha=0.35, size=2.5, ax=axes[2])
axes[2].set_title('Age distribution by diagnosis')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'full_demographics_distribution.png', dpi=180, bbox_inches='tight')
plt.show()

## 2. Distribution of Subjects in the Provided Folds

The fold manifest is built directly from `splits/Fold_*`. Each subject is present once in every fold, assigned to one of training, validation, or test. For descriptive signal and feature analysis, the 100 unique subjects are used once each.

In [ ]:
manifest_rows = []
for path in sorted(SPLITS_DIR.glob('Fold_*/*/*_01.txt')):
    subject_id = path.stem.rsplit('_', 1)[0]
    manifest_rows.append({
        'fold': path.parts[-3],
        'split': path.parts[-2],
        'subject_id': subject_id,
        'group': 'Patient' if 'Pt' in subject_id else 'Control',
        'study': subject_id[:2],
        'filepath': str(path),
    })
manifest = pd.DataFrame(manifest_rows)
unique_subject_files = (manifest.sort_values(['subject_id', 'fold', 'split'])
                        .drop_duplicates('subject_id')
                        .reset_index(drop=True))
fold_subject_ids = set(unique_subject_files['subject_id'])
fold_demographics = demographics[demographics['ID'].isin(fold_subject_ids)].copy()

assert manifest.groupby('fold')['subject_id'].nunique().eq(100).all()
assert len(unique_subject_files) == 100
assert len(fold_demographics) == 100

fold_split_counts = (manifest.groupby(['fold', 'split', 'group'], as_index=False)
                     .size().rename(columns={'size': 'subjects'}))
fold_demographic_summary = demographic_summary(fold_demographics)
fold_demographic_summary.to_csv(OUTPUT_DIR / 'fold_demographic_summary.csv', index=False)
fold_split_counts.to_csv(OUTPUT_DIR / 'fold_split_counts.csv', index=False)
print('Unique subjects present in the fold system:')
display(fold_demographic_summary)
print('Per-fold assignment counts:')
display(fold_split_counts.pivot_table(index=['fold', 'split'], columns='group', values='subjects'))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
sns.countplot(data=fold_demographics, x='Group', hue='Group', palette=PALETTE, legend=False, ax=axes[0])
axes[0].set_title('Fold population by diagnosis')
axes[0].set_ylabel('Unique subjects')
sns.countplot(data=fold_demographics, x='Group', hue='Gender', palette=['#204251', '#8C9FB1'], ax=axes[1])
axes[1].set_title('Fold population sex distribution')
sns.barplot(data=fold_split_counts, x='fold', y='subjects', hue='split', palette=['#204251', '#8C9FB1', '#C2D0DC'], ax=axes[2])
axes[2].set_title('Subjects per split and class')
axes[2].set_ylabel('Subjects per class')
axes[2].set_xlabel('Fold (each bar is per class)')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'fold_subject_distribution.png', dpi=180, bbox_inches='tight')
plt.show()

## 3. Patient and Control Signal Examples

Each text file contains time, eight VGRF sensors under each foot, and total left/right force. The example figure shows the total-force time series for one deterministically selected patient and one control. Shaded endpoint regions indicate the optional 5-second trimming evaluated later; the untrimmed variant retains these samples.

In [ ]:
def load_signal(filepath):
    return pd.read_csv(filepath, sep=r'\s+', header=None, names=SIGNAL_COLUMNS)

examples = (unique_subject_files.sort_values('subject_id')
            .groupby('group', as_index=False).first())
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=False)
for ax, (_, row) in zip(axes, examples.iterrows()):
    signal = load_signal(row['filepath'])
    t = signal['Time'].to_numpy()
    duration = float(t[-1])
    ax.plot(t, signal['TotalLeft'], lw=0.85, color='#2F586E', label='Left total force')
    ax.plot(t, signal['TotalRight'], lw=0.85, color='#B24545', alpha=0.85, label='Right total force')
    ax.axvspan(t[0], t[0] + TRIM_SECONDS, color='#C2D0DC', alpha=0.65, label='Optional removed endpoint')
    ax.axvspan(t[-1] - TRIM_SECONDS, t[-1], color='#C2D0DC', alpha=0.65)
    ax.set_title(f"{row['group']}: {row['subject_id']} ({duration:.1f} s)", loc='left')
    ax.set_ylabel('VGRF (N)')
    ax.legend(loc='upper right', ncol=3, fontsize=9)
axes[-1].set_xlabel('Time (seconds)')
fig.suptitle('Example walking recordings and optional endpoint trimming regions', y=1.01, fontweight='bold')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'example_signals_with_trim_regions.png', dpi=180, bbox_inches='tight')
plt.show()
display(examples[['subject_id', 'group', 'study']])

## 4. TSFresh Feature Extraction and Endpoint Sensitivity Analysis

The deliverable suggests TSFresh features on 5-second windows with a 2.5-second step. This notebook uses `MinimalFCParameters`, a compact and reproducible TSFresh set that keeps this unsupervised deliverable practical while satisfying the requested TSFresh computation.

For every subject and preprocessing variant:

1. Extract TSFresh features from overlapping windows using all nine left-foot channels and all nine right-foot channels.
2. Separate the left and right window feature vectors.
3. Aggregate every resulting window feature per subject using **mean, standard deviation, skewness, and kurtosis**, as required.
4. Form the combined-feet representation by concatenating the left- and right-foot subject vectors.

**Why compare endpoint trimming?** Walking recordings can contain initiation and stopping transients. Removing 5 seconds from each endpoint may concentrate the feature representation on steadier gait. However, trimming also discards data, so both variants are computed and visualized. This preprocessing comparison is an analysis choice here, not a method claimed from a paper.

In [ ]:
FC_PARAMETERS = MinimalFCParameters()
ALL_KINDS = LEFT_KINDS + RIGHT_KINDS

def trim_signal(signal, trim_seconds):
    if trim_seconds == 0:
        return signal.copy()
    n = int(trim_seconds * SAMPLE_RATE)
    if len(signal) <= 2 * n + WINDOW_SAMPLES:
        raise ValueError('Signal is too short for the requested trimming and a complete window.')
    return signal.iloc[n:-n].reset_index(drop=True)

def to_tsfresh_windows(signal):
    frames = []
    n_windows = 0
    for n_windows, start in enumerate(range(0, len(signal) - WINDOW_SAMPLES + 1, STEP_SAMPLES), start=1):
        block = signal.iloc[start:start + WINDOW_SAMPLES]
        for kind in ALL_KINDS:
            frames.append(pd.DataFrame({
                'window_id': f'w{n_windows:03d}',
                'time': np.arange(WINDOW_SAMPLES),
                'kind': kind,
                'value': block[kind].to_numpy(),
            }))
    if not frames:
        raise ValueError('No complete 5-second window can be formed.')
    return pd.concat(frames, ignore_index=True), n_windows

def aggregate_window_features(window_features, kinds, prefix):
    side_cols = [column for column in window_features.columns if column.split('__', 1)[0] in kinds]
    side = window_features[side_cols]
    aggregates = {
        'mean': side.mean(axis=0),
        'std': side.std(axis=0, ddof=0),
        'skewness': side.skew(axis=0),
        'kurtosis': side.kurt(axis=0),
    }
    values = {}
    for statistic, series in aggregates.items():
        for feature_name, value in series.fillna(0.0).items():
            values[f'{prefix}__{feature_name}__window_{statistic}'] = float(value)
    return values

def compute_subject_feature_matrix(trim_seconds, variant_name, recompute=RECOMPUTE_FEATURES):
    matrix_path = OUTPUT_DIR / f'tsfresh_subject_features_{variant_name}.csv'
    window_path = OUTPUT_DIR / f'window_counts_{variant_name}.csv'
    if matrix_path.exists() and window_path.exists() and not recompute:
        return pd.read_csv(matrix_path), pd.read_csv(window_path)

    rows = []
    window_rows = []
    for idx, subject in unique_subject_files.iterrows():
        signal = trim_signal(load_signal(subject['filepath']), trim_seconds)
        long_windows, n_windows = to_tsfresh_windows(signal)
        window_features = extract_features(
            long_windows,
            column_id='window_id', column_sort='time',
            column_kind='kind', column_value='value',
            default_fc_parameters=FC_PARAMETERS,
            disable_progressbar=True, show_warnings=False, n_jobs=0,
        )
        impute(window_features)
        row = {'subject_id': subject['subject_id'], 'group': subject['group'], 'study': subject['study']}
        row.update(aggregate_window_features(window_features, LEFT_KINDS, 'left'))
        row.update(aggregate_window_features(window_features, RIGHT_KINDS, 'right'))
        rows.append(row)
        window_rows.append({'subject_id': subject['subject_id'], 'group': subject['group'], 'windows': n_windows, 'samples_used': len(signal)})
        if (idx + 1) % 20 == 0:
            print(f'{variant_name}: processed {idx + 1}/100 subjects')

    matrix = pd.DataFrame(rows).sort_values('subject_id').reset_index(drop=True)
    window_counts = pd.DataFrame(window_rows).sort_values('subject_id').reset_index(drop=True)
    matrix.to_csv(matrix_path, index=False)
    window_counts.to_csv(window_path, index=False)
    return matrix, window_counts

In [ ]:
features_untrimmed, windows_untrimmed = compute_subject_feature_matrix(0.0, 'untrimmed')
features_trimmed, windows_trimmed = compute_subject_feature_matrix(TRIM_SECONDS, 'trim_5s_each_end')

def scope_columns(matrix, scope):
    if scope == 'Left foot':
        return [c for c in matrix.columns if c.startswith('left__')]
    if scope == 'Right foot':
        return [c for c in matrix.columns if c.startswith('right__')]
    if scope == 'Combined feet':
        return [c for c in matrix.columns if c.startswith(('left__', 'right__'))]
    raise ValueError(scope)

feature_extraction_summary = pd.DataFrame([
    {
        'preprocessing': 'No endpoint removal',
        'subjects': len(features_untrimmed),
        'mean_windows_per_subject': windows_untrimmed['windows'].mean(),
        'min_windows': windows_untrimmed['windows'].min(),
        'max_windows': windows_untrimmed['windows'].max(),
        'left_features': len(scope_columns(features_untrimmed, 'Left foot')),
        'right_features': len(scope_columns(features_untrimmed, 'Right foot')),
        'combined_features': len(scope_columns(features_untrimmed, 'Combined feet')),
    },
    {
        'preprocessing': 'Remove 5 s at each endpoint',
        'subjects': len(features_trimmed),
        'mean_windows_per_subject': windows_trimmed['windows'].mean(),
        'min_windows': windows_trimmed['windows'].min(),
        'max_windows': windows_trimmed['windows'].max(),
        'left_features': len(scope_columns(features_trimmed, 'Left foot')),
        'right_features': len(scope_columns(features_trimmed, 'Right foot')),
        'combined_features': len(scope_columns(features_trimmed, 'Combined feet')),
    },
]).round({'mean_windows_per_subject': 1})
feature_extraction_summary.to_csv(OUTPUT_DIR / 'feature_extraction_summary.csv', index=False)
print('Feature extraction summary for both endpoint choices:')
display(feature_extraction_summary)

window_compare = windows_untrimmed[['subject_id', 'windows']].merge(
    windows_trimmed[['subject_id', 'windows']], on='subject_id', suffixes=('_untrimmed', '_trimmed'))
window_compare['windows_removed'] = window_compare['windows_untrimmed'] - window_compare['windows_trimmed']
window_compare.to_csv(OUTPUT_DIR / 'endpoint_trim_window_comparison.csv', index=False)
print('Windows removed by the optional 5-second endpoint trim:')
display(window_compare['windows_removed'].describe().to_frame().T.round(2))

## 5. Z-Score Normalization Followed by PCA and t-SNE

The scaling is explicitly inside the reduction function below. Constant feature columns are removed first because they carry no clustering information and cannot be meaningfully standardized. Every retained feature is then standardized to zero mean and unit population standard deviation before PCA or t-SNE.

In [ ]:
SCOPES = ['Right foot', 'Left foot', 'Combined feet']
VARIANTS = {
    'No endpoint removal': features_untrimmed,
    'Remove 5 s at each endpoint': features_trimmed,
}

def standardized_feature_array(matrix, scope):
    columns = scope_columns(matrix, scope)
    x = matrix[columns].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    variable_columns = x.columns[x.var(axis=0, ddof=0) > 0]
    x = x[variable_columns]
    scaler = StandardScaler()
    x_z = scaler.fit_transform(x)
    audit = {
        'scope': scope,
        'input_features': len(columns),
        'retained_nonconstant_features': len(variable_columns),
        'max_abs_scaled_mean': np.abs(x_z.mean(axis=0)).max(),
        'max_abs_scaled_std_minus_one': np.abs(x_z.std(axis=0) - 1.0).max(),
    }
    return x_z, audit

normalization_audit = []
for variant, matrix in VARIANTS.items():
    for scope in SCOPES:
        _, audit = standardized_feature_array(matrix, scope)
        audit['preprocessing'] = variant
        normalization_audit.append(audit)
normalization_audit = pd.DataFrame(normalization_audit)
normalization_audit.to_csv(OUTPUT_DIR / 'zscore_normalization_audit.csv', index=False)
print('Z-score normalization audit before dimensionality reduction:')
display(normalization_audit.round(10))

def projection(matrix, scope, method):
    x_z, _ = standardized_feature_array(matrix, scope)
    if method == 'PCA':
        model = PCA(n_components=2, random_state=42)
        coords = model.fit_transform(x_z)
        subtitle = f'PC1 {model.explained_variance_ratio_[0]*100:.1f}%, PC2 {model.explained_variance_ratio_[1]*100:.1f}%'
    elif method == 't-SNE':
        n_components = min(50, x_z.shape[0] - 1, x_z.shape[1])
        x_initial = PCA(n_components=n_components, random_state=42).fit_transform(x_z)
        model = TSNE(n_components=2, perplexity=20, init='pca', learning_rate='auto', max_iter=1000, random_state=42)
        coords = model.fit_transform(x_initial)
        subtitle = 'perplexity = 20, random_state = 42'
    else:
        raise ValueError(method)
    result = pd.DataFrame({'x': coords[:, 0], 'y': coords[:, 1], 'group': matrix['group'], 'subject_id': matrix['subject_id']})
    return result, subtitle

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for row, (variant, matrix) in enumerate(VARIANTS.items()):
    for col, scope in enumerate(SCOPES):
        coords, subtitle = projection(matrix, scope, 'PCA')
        sns.scatterplot(data=coords, x='x', y='y', hue='group', palette=PALETTE, s=58, alpha=0.82, ax=axes[row, col])
        axes[row, col].set_title(f'{scope} | {subtitle}', fontsize=10)
        axes[row, col].set_xlabel('PC1')
        axes[row, col].set_ylabel('PC2')
        axes[row, col].legend(title='', fontsize=8)
    axes[row, 0].annotate(variant, xy=(-0.27, 0.5), xycoords='axes fraction', rotation=90, va='center', fontsize=11, fontweight='bold')
fig.suptitle('PCA after z-score normalization: endpoint trimming sensitivity', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'pca_three_scopes_trim_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for row, (variant, matrix) in enumerate(VARIANTS.items()):
    for col, scope in enumerate(SCOPES):
        coords, subtitle = projection(matrix, scope, 't-SNE')
        sns.scatterplot(data=coords, x='x', y='y', hue='group', palette=PALETTE, s=58, alpha=0.82, ax=axes[row, col])
        axes[row, col].set_title(f'{scope} | {subtitle}', fontsize=10)
        axes[row, col].set_xlabel('t-SNE dimension 1')
        axes[row, col].set_ylabel('t-SNE dimension 2')
        axes[row, col].legend(title='', fontsize=8)
    axes[row, 0].annotate(variant, xy=(-0.27, 0.5), xycoords='axes fraction', rotation=90, va='center', fontsize=11, fontweight='bold')
fig.suptitle('t-SNE after z-score normalization: endpoint trimming sensitivity', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'tsne_three_scopes_trim_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

## Feature Approaches Permitted by the Deliverable

The PDF explicitly permits alternatives to TSFresh features, different step sizes, and feature-combination methods other than concatenation. This notebook keeps the required analysis interpretable and directly comparable by using TSFresh plus concatenation, while varying only the endpoint treatment. Possible follow-up exploratory feature families, still relevant to the deliverable but not necessary for this report, include gait-event timing, stance/swing variability, force asymmetry, center-of-pressure trajectories, or wavelet descriptors.

The existing `data_analysis.py` informed the general plot structure, but the notebook is self-contained and corrects the subject-selection logic by using all 100 unique subjects represented in the supplied five-fold system, including validation assignments.

## Sources and Endpoint-Trimming Note

- Required tasks: `../Deliverable_1_List_of_Task.pdf`.
- Signal column structure, sensor interpretation, walk identifier, and 100 Hz sampling rate: `format.txt` supplied with the data.
- The 5-second endpoint-trimming comparison in this notebook is not cited as a rule from a research paper. It is an explicitly labeled sensitivity analysis for possible initiation/stopping effects, and the untrimmed required analysis is retained alongside it.

In [ ]:
print('Generated analysis files:')
for file in sorted(OUTPUT_DIR.glob('*')):
    print('-', file.name)
print('\nDeliverable 1 notebook complete: no supervised model results were computed or reported.')